In [ ]:
pip install findspark

In [1]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


Successfully connected to HYPE on port 1521.


In [2]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [3]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
osql_jar_path = r"file:///C:/spark_drivers/ojdbc17.jar"

spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{osql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

Findspark initialized successfully.
Spark session created.


In [14]:
# SQL Server connection properties
# cannot make partition with char column
from pyspark.sql import functions as sf
import time
sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=sttwk_2025a_par;encrypt=true;trustServerCertificate=true"
sqlserver_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchSize": "1000"
}
# --- Configuration (reuse previous variables) ---
#sql_server_url = "your_sql_server_url"
oracle_url = "your_oracle_url"
# ... credentials ...
source_view_name = "v_sttwk_2025_par" # This view has YEAR_WEEK column for partitioning
oracle_destination_table = "STTWK"   # This table does NOT have YEAR_WEEK

partition_column_name = "yr_wk"
# Assume year_week_list has 181 specific values:
# year_week_list = ['202201', '202202', ... ]
year_week_list = [str(y) + str(w).zfill(2) for y in range(2022, 2026) for w in range(1, 53)] # This generates too many, but illustrates the format
# Create the specific predicates list
predicates = [f"{partition_column_name} = '{yw_value}'" for yw_value in year_week_list]


# 2. Read data from SQL Server using the 181 predicates
#    We define the columns explicitly to IGNORE the year_week column
#    that is only used for the WHERE clause (predicates)
columns_to_select = "branch_id, sp_id, custd_id, prod_id, sls_id, yr, mth, wk, net_qty, net_amt, grs_qty, grs_amt, rtn_qty, rtn_amt"
partition_column_name = "yr_wk"
# Start the timer
start_time = time.time()

# Parameters for parallel read (adjust as needed for your table and data)
# The partitionColumn must be a numeric, date, or timestamp column.
#partition_column = "week" 
#lower_bound = 1
#upper_bound = 13 # Max ID in your table, e.g.
#num_read_partitions = 13 # Read in 10 partitions
    #.option("url", sqlserver_jdbc_url) \
    #.option("partitionColumn", partition_column) \
    #.option("lowerBound", lower_bound) \
    #.option("upperBound", upper_bound) \
    #.option("numPartitions", num_read_partitions) \
df_partitioned = spark.read.format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", source_view_name) \
    .option("fetchsize", 1000) \
    .option("user", sqlserver_properties["user"]) \
    .option("password", sqlserver_properties["password"]) \
    .option("customSql", f"SELECT {columns_to_select} FROM {source_view_name}") \
    .option("driver", sqlserver_properties["driver"]) \
    .load(predicates=predicates) 
# Note: When using predicates with custom selects, ensure the source table/view in dbtable 
# is correct and the predicates column is available in the source (which it is).
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions.")
df_partitioned.printSchema() # Verify year_week is not in the schema
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")
# Optional: Check the number of partitions created
#print(f"DataFrame has {sql_server_df.rdd.getNumPartitions()} partitions after read.")


DataFrame has 1 partitions.
root
 |-- branch_id: string (nullable = true)
 |-- sp_id: string (nullable = true)
 |-- custd_id: string (nullable = true)
 |-- prod_id: string (nullable = true)
 |-- sls_id: string (nullable = true)
 |-- yr: string (nullable = true)
 |-- mth: string (nullable = true)
 |-- wk: string (nullable = true)
 |-- yr_wk: string (nullable = true)
 |-- net_qty: decimal(12,4) (nullable = true)
 |-- net_amt: decimal(18,4) (nullable = true)
 |-- grs_qty: decimal(12,4) (nullable = true)
 |-- grs_amt: decimal(18,4) (nullable = true)
 |-- rtn_qty: decimal(10,2) (nullable = true)
 |-- rtn_amt: decimal(15,2) (nullable = true)

Total time taken for transfer: 0.33 seconds


In [17]:
# 10 seconds ? no

# Start the timer
start_time = time.time()

# Option A: Repartition by the character column (hash partitioning)
# The data associated with each unique value in 'char_column' will go to a single partition.
#df_to_write = sql_server_df.repartition(181, "yr_wk")
df_to_write = df_partitioned.repartition(181, "yr_wk")
# Option B: Repartition generally (even hash distribution)
# Use this if you don't care about co-locating data by the char column, 
# or if the data in 'your_char_column_name' is very skewed.
# df_to_write = sql_server_df.repartition(10) 

print(f"DataFrame has {df_to_write.rdd.getNumPartitions()} partitions before write.")
predicates = [f"{partition_column_name} = '{yw_value}'" for yw_value in year_week_list]
columns_to_select = "branch_id, sp_id, custd_id, prod_id, sls_id, yr, mth, wk, net_qty, net_amt, grs_qty, grs_amt, rtn_qty, rtn_amt"

# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

DataFrame has 181 partitions before write.
Total time taken for transfer: 255.76 seconds


In [21]:
# Drop the temporary 'yr_wk' column before writing to Oracle
df_to_write = df_partitioned.drop("yr_wk")


In [ ]:
# --- RERUN THIS CELL --- this cell only for backup if above is error..it is similar actually
# Assuming 'year_week_list' and 'predicates' are already defined

# Read data from SQL Server using the 181 predicates with the standard dbtable option
df_partitioned = spark.read.format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", source_view_name) \
    .option("fetchsize", 1000) \
    .option("driver", sqlserver_properties["driver"]) \
    .load(predicates=predicates) 

# Verify the partitioning worked
print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions.")
df_partitioned.printSchema() 

# Drop the temporary 'yr_wk' column before writing to Oracle
df_to_write_to_oracle = df_partitioned.drop("yr_wk")

print(f"Schema after dropping yr_wk column:")
df_to_write_to_oracle.printSchema()


In [22]:
# osql connection properties
# Start the timer
start_time = time.time()
# Connection details for osql (Destination)
oracle_host = "localhost"
oracle_port = "1521"
oracle_service_name = "XE" # Standard service name for XE
oracle_user = "system"        
oracle_password = "D@1ry$2014" 
oracle_table = "STTWK"        
oracle_driver_class = "oracle.jdbc.driver.OracleDriver" 

# This is the crucial line: Make sure the URL is formatted exactly like this:
oracle_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service_name}"


oracle_jdbc_url =  f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service_name}"
 
 
# Write the repartitioned DataFrame to the osql table
df_to_write.write.format("jdbc") \
    .option("url", oracle_url) \
    .option("dbtable", oracle_table) \
    .option("user", oracle_user) \
    .option("password", oracle_password) \
    .option("driver", oracle_driver_class) \
    .option("batchsize", "10000") \
    .mode("append") \
    .save()


print("Data successfully inserted into osql table using 181 parallel processes.")
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")    

Data successfully inserted into osql table using 181 parallel processes.
Total time taken for transfer: 548.32 seconds


In [ ]:
# Assuming 'spark' session is still active from Cell 1
# Start the timer
start_time = time.time()

# Connection details
osql_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service_name}"
osql_user = "system"
osql_password = "D@1ry$2014" 
oracle_driver_class = "oracle.jdbc.driver.OracleDriver"

# Define the full, raw SQL query.
# IMPORTANT: Remove the AS data_count alias, as Spark handles that internally now.
full_sql_query = "select yr,wk,sum(net_qty) as net_qty from sttwk group by yr,wk"

# Read the data using the 'query' option instead of 'dbtable'
df_sample = spark.read.format("jdbc").options(
    url=osql_url,
    query=full_sql_query,  # Use 'query' instead of 'dbtable'
    user=osql_user,
    password=osql_password,
    driver=oracle_driver_class
).load()

print("Displaying aggregated records from Oracle table 'sttwk':")

# Use .show() to display the records (ordering them in Spark)
df_sample.orderBy("yr", "wk").show(n=1000, truncate=False)

# To get the count of the sample:
print(f"Number of sample rows read: {df_sample.count()}")

# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")
